## SQL Analysis
This notebook contains the required SQL queries for our final project. These queries are used to validate the structure and quality of the merged Spotify and Last.fm dataset, measure overlap between the two sources, and identify artist- and song-level patterns in popularity and listener engagement. The SQL analysis supports our broader project goal of helping A&R and marketing teams understand which track characteristics and engagement signals are most associated with Spotify popularity. The queries include joins, subqueries, window functions, and grouped aggregations required by the rubric.

In [1]:
import pandas as pd
import sqlite3
import re

conn = sqlite3.connect("music_project.sqlite")

In [2]:
tracks_df = pd.read_csv("lastfm_tracks.csv")
tracks_df

,artist,track_name,listeners,playcount,artist_clean,track_clean
0,PinkPantheress,Pain,1851335,34921372,pinkpantheress,pain
1,PinkPantheress,Boy's a liar Pt. 2,1565235,22250991,pinkpantheress,boy's a liar pt. 2
2,PinkPantheress,I Must Apologise,1169061,15481727,pinkpantheress,i must apologise
3,PinkPantheress,Break It Off - Bonus,1064063,11134317,pinkpantheress,break it off - bonus
4,PinkPantheress,Attracted to You,1034687,17174924,pinkpantheress,attracted to you
...,...,...,...,...,...,...
995,Zara Larsson,Can't Tame Her,224330,2654492,zara larsson,can't tame her
996,Zara Larsson,Pretty Ugly,201279,2145159,zara larsson,pretty ugly
997,Zara Larsson,Blue Moon,200710,1661577,zara larsson,blue moon
998,Zara Larsson,I Would Like,199831,1417164,zara larsson,i would like


In [3]:
spotify_df = pd.read_csv("dataset.csv")
spotify_df

,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.7150,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.2670,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.1200,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.1430,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.1670,119.949,4,acoustic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113995,113995,2C3TZjDRiAzdyViavDJ217,Rainy Lullaby,#mindfulness - Soft Rain for Mindful Meditatio...,Sleep My Little Boy,21,384999,False,0.172,0.2350,...,-16.393,1,0.0422,0.6400,0.928000,0.0863,0.0339,125.995,5,world-music
113996,113996,1hIz5L4IB9hN3WRYPOCGPw,Rainy Lullaby,#mindfulness - Soft Rain for Mindful Meditatio...,Water Into Light,22,385000,False,0.174,0.1170,...,-18.318,0,0.0401,0.9940,0.976000,0.1050,0.0350,85.239,4,world-music
113997,113997,6x8ZfSoqDjuNa5SVP5QjvX,Cesária Evora,Best Of,Miss Perfumado,22,271466,False,0.629,0.3290,...,-10.895,0,0.0420,0.8670,0.000000,0.0839,0.7430,132.378,4,world-music
113998,113998,2e6sXL2bYv4bSz6VTdnfLs,Michael W. Smith,Change Your World,Friends,41,283893,False,0.587,0.5060,...,-10.889,1,0.0297,0.3810,0.000000,0.2700,0.4130,135.960,4,world-music


In [4]:
def clean_text(x):
    x = str(x).lower().strip()
    x = re.sub(r"\(.*?\)", "", x)          # remove ( ... )
    x = re.sub(r"\[.*?\]", "", x)          # remove [ ... ]
    x = re.sub(r"\b(feat|ft)\b.*", "", x)  # remove feat/ft and after
    x = re.sub(r"\s*-\s*.*", "", x)        # remove " - remix" etc
    x = re.sub(r"[^a-z0-9\s]", "", x)      # remove punctuation
    x = re.sub(r"\s+", " ", x).strip()     # normalize spaces
    return x

# Last.fm keys (your columns are artist + track_name)
tracks_df["artist_clean"] = tracks_df["artist"].apply(clean_text)
tracks_df["track_clean"]  = tracks_df["track_name"].apply(clean_text)

# Spotify keys (your columns are artists + track_name)
spotify_df["artist_clean"] = spotify_df["artists"].apply(clean_text)
spotify_df["track_clean"]  = spotify_df["track_name"].apply(clean_text)

In [5]:
tracks_df.to_sql("lastfm", conn, if_exists="replace", index=False)
spotify_df.to_sql("spotify", conn, if_exists="replace", index=False)

114000

In [6]:
def read_sql(q):
    return pd.read_sql(q, conn)

def exec_sql(q):
    conn.execute(q)
    conn.commit()

In [7]:
# Query 1: Preview the Spotify dataset
# WHAT: Displays a small sample of rows from the Spotify table.
# HOW: Uses SELECT with LIMIT to inspect the structure and values.
# WHY: This confirms that the Spotify data loaded correctly and helps verify the main variables available for analysis.

pd.read_sql("""
SELECT *
FROM spotify
LIMIT 10
""", conn)

,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre,artist_clean,track_clean
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,0,0.676,0.4610,...,0.1430,0.0322,0.000001,0.3580,0.7150,87.917,4,acoustic,gen hoshino,comedy
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,0,0.420,0.1660,...,0.0763,0.9240,0.000006,0.1010,0.2670,77.489,4,acoustic,ben woodward,ghost
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,0,0.438,0.3590,...,0.0557,0.2100,0.000000,0.1170,0.1200,76.332,4,acoustic,ingrid michaelsonzayn,to begin again
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,0,0.266,0.0596,...,0.0363,0.9050,0.000071,0.1320,0.1430,181.740,3,acoustic,kina grannis,cant help falling in love
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,0,0.618,0.4430,...,0.0526,0.4690,0.000000,0.0829,0.1670,119.949,4,acoustic,chord overstreet,hold on
5,5,01MVOl9KtVTNfFiBU9I7dc,Tyrone Wells,Days I Will Remember,Days I Will Remember,58,214240,0,0.688,0.4810,...,0.1050,0.2890,0.000000,0.1890,0.6660,98.017,4,acoustic,tyrone wells,days i will remember
6,6,6Vc5wAMmXdKIAM7WUoEb7N,A Great Big World;Christina Aguilera,Is There Anybody Out There?,Say Something,74,229400,0,0.407,0.1470,...,0.0355,0.8570,0.000003,0.0913,0.0765,141.284,3,acoustic,a great big worldchristina aguilera,say something
7,7,1EzrEOXmMH3G43AXT1y7pA,Jason Mraz,We Sing. We Dance. We Steal Things.,I'm Yours,80,242946,0,0.703,0.4440,...,0.0417,0.5590,0.000000,0.0973,0.7120,150.960,4,acoustic,jason mraz,im yours
8,8,0IktbUcnAGrvD03AWnz3Q8,Jason Mraz;Colbie Caillat,We Sing. We Dance. We Steal Things.,Lucky,74,189613,0,0.625,0.4140,...,0.0369,0.2940,0.000000,0.1510,0.6690,130.088,4,acoustic,jason mrazcolbie caillat,lucky
9,9,7k9GuJYLp2AzqokyEdwEw2,Ross Copperman,Hunger,Hunger,56,205594,0,0.442,0.6320,...,0.0295,0.4260,0.004190,0.0735,0.1960,78.899,4,acoustic,ross copperman,hunger


In [8]:
# Query 2: Preview the Last.fm dataset
# WHAT: Displays a small sample of rows from the Last.fm table.
# HOW: Uses SELECT with LIMIT to inspect the structure and values.
#WHY: This confirms that the Last.fm data loaded correctly and helps verify the engagement variables used later in the notebook. 

pd.read_sql("""
SELECT *
FROM lastfm
LIMIT 10
""", conn)

,artist,track_name,listeners,playcount,artist_clean,track_clean
0,PinkPantheress,Pain,1851335,34921372,pinkpantheress,pain
1,PinkPantheress,Boy's a liar Pt. 2,1565235,22250991,pinkpantheress,boys a liar pt 2
2,PinkPantheress,I Must Apologise,1169061,15481727,pinkpantheress,i must apologise
3,PinkPantheress,Break It Off - Bonus,1064063,11134317,pinkpantheress,break it off
4,PinkPantheress,Attracted to You,1034687,17174924,pinkpantheress,attracted to you
5,PinkPantheress,Illegal,1018907,11267265,pinkpantheress,illegal
6,PinkPantheress,Stateside + Zara Larsson,860462,11683964,pinkpantheress,stateside zara larsson
7,PinkPantheress,Just for Me,844071,10217944,pinkpantheress,just for me
8,PinkPantheress,Tonight,775719,11297803,pinkpantheress,tonight
9,PinkPantheress,Boy's a liar,746273,6539323,pinkpantheress,boys a liar


In [9]:
#Query 3: Average popularity by artist
# WHAT: Calculates each artist's average Spotify popularity and number of songs in the dataset.
# HOW: Groups rows by artist and computes mean popularity and track count.
# WHY: This helps identify whether stronger artist performance reflects consistently popular catalogs or only a few songs.

pd.read_sql("""
SELECT
artists,
AVG(popularity) AS avg_popularity,
COUNT(*) AS song_count
FROM spotify
GROUP BY artists
ORDER BY avg_popularity DESC
LIMIT 10
""", conn)

,artists,avg_popularity,song_count
0,Sam Smith;Kim Petras,100.0,2
1,Bizarrap;Quevedo,99.0,1
2,Manuel Turizo,98.0,4
3,Bad Bunny;Chencho Corleone,97.0,4
4,Bad Bunny;Bomba Estéreo,94.5,4
5,Joji,94.0,1
6,Beyoncé,93.0,1
7,Rema;Selena Gomez,92.0,1
8,Harry Styles,92.0,3
9,Rauw Alejandro;Lyanno;Brray,91.0,2


This query is helpful because it moves beyond song-level analysis and looks at artist-level consistency. For the target audience, this matters because a label may care not only about one breakout track, but also whether an artist appears to perform strongly across multiple songs.

In [10]:
# Query 4: Compare audio features for above- and below-average popularity songs
# WHAT: Compares average audio features for songs above and below the dataset-wide average popularity.
# HOW: Uses a subquery to compute average popularity, then groups songs into two categories with CASE.
#WHY: This helps show whether stronger-performing songs have noticeably different audio profiles.

pd.read_sql("""
SELECT
  CASE
    WHEN popularity > (SELECT AVG(popularity) FROM spotify) THEN 'Above Average Popularity'
    ELSE 'Below Average Popularity'
  END AS popularity_group,
  AVG(danceability) AS avg_danceability,
  AVG(energy) AS avg_energy,
  AVG(tempo) AS avg_tempo,
  AVG(loudness) AS avg_loudness,
  COUNT(*) AS track_count
FROM spotify
GROUP BY popularity_group
""", conn)

,popularity_group,avg_danceability,avg_energy,avg_tempo,avg_loudness,track_count
0,Above Average Popularity,0.576144,0.636887,122.506806,-8.024592,58418
1,Below Average Popularity,0.556979,0.646108,121.770553,-8.505287,55582


In [11]:
# Query 5: Join Spotify and Last.fm on matching artist and track names
# WHAT: Combines Spotify audio features with Last.fm engagement metrics for matched songs.
#HOW: Uses an INNER JOIN on shared artist and track identifiers.
# WHY: This creates the core merged dataset needed to compare platform popularity with listener activity.
pd.read_sql("""
SELECT
s.artists,
s.track_name,
s.popularity,
l.listeners,
l.playcount
FROM spotify s
JOIN lastfm l
ON s.track_name = l.track_name
AND s.artists = l.artist
LIMIT 20
""", conn)

,artists,track_name,popularity,listeners,playcount
0,The Neighbourhood,Daddy Issues,87,1835394,29887666
1,The Neighbourhood,Softcore,86,1499043,25594547
2,The Neighbourhood,Sweater Weather,93,2897217,48497000
3,The Neighbourhood,You Get Me So High,83,1376338,22668387
4,Nirvana,Smells Like Teen Spirit,83,3998444,40954484
5,The Killers,Mr. Brightside,2,3599455,49369280
6,Red Hot Chili Peppers,Californication,2,2997449,27590499
7,The Killers,When You Were Young,5,2305587,22582654
8,The Killers,Mr. Brightside,2,3599455,49369280
9,The Killers,Mr. Brightside,0,3599455,49369280


This join is central to the project because it allows the analysis to connect two different dimensions of music performance: how songs sound and how listeners engage with them. At the same time, the join is limited by metadata differences across platforms, so the matched sample should be interpreted as a filtered overlap rather than a complete universe of songs.

In [12]:
# Query 6: Songs with strong Spotify popularity and Last.fm engagement
# WHAT: Identifies matched songs that perform strongly across both Spotify and Last.fm.
# HOW: Joins the datasets and sorts or filters based on popularity and engagement measures.
# WHY: This helps highlight songs with cross-platform traction rather than strength in only one source.

pd.read_sql("""
SELECT
s.artists,
s.track_name,
s.popularity,
l.playcount
FROM spotify s
JOIN lastfm l
ON s.track_name = l.track_name
AND s.artists = l.artist
WHERE s.popularity > 70
ORDER BY l.playcount DESC
LIMIT 20
""", conn)

,artists,track_name,popularity,playcount
0,Radiohead,Creep,85,58398696
1,Radiohead,Creep,85,58398696
2,Radiohead,Creep,85,58398696
3,Radiohead,Creep,85,58398696
4,Frank Ocean,Pink + White,85,57862066
5,Arctic Monkeys,505,88,56138091
6,Arctic Monkeys,505,88,56138091
7,Arctic Monkeys,505,88,56138091
8,Radiohead,No Surprises,82,53597143
9,Radiohead,No Surprises,82,53597143


This query is especially useful for the target audience because strong cross-platform performance may be more meaningful than a high score in only one dataset. A song that performs well across both Spotify and Last.fm may indicate broader listener traction.

In [13]:
# Query 7: Songs with popularity above the dataset average
# WHAT: Returns songs whose Spotify popularity is higher than the overall average.
# HOW: Uses a subquery to calculate the dataset-wide average popularity, then filters the rows above that threshold.
# WHY: This isolates stronger-performing songs and provides a simple benchmark for comparison.

pd.read_sql("""
SELECT
artists,
track_name,
popularity
FROM spotify
WHERE popularity >
(
SELECT AVG(popularity)
FROM spotify
)
ORDER BY popularity DESC
""", conn)

,artists,track_name,popularity
0,Sam Smith;Kim Petras,Unholy (feat. Kim Petras),100
1,Sam Smith;Kim Petras,Unholy (feat. Kim Petras),100
2,Bizarrap;Quevedo,"Quevedo: Bzrp Music Sessions, Vol. 52",99
3,David Guetta;Bebe Rexha,I'm Good (Blue),98
4,David Guetta;Bebe Rexha,I'm Good (Blue),98
...,...,...,...
58413,Brian Doerksen,Come Now Is The Time To Worship,34
58414,Phil Wickham,Tethered,34
58415,Hillsong Worship,O Holy Night,34
58416,Phil Wickham,House Of The Lord - Acoustic,34


This query is useful because it identifies songs that outperform the typical track in the dataset, creating a more meaningful comparison group than simply looking at all songs together.

In [14]:
# Query 8: Artists with more songs than the average artist
# WHAT: Identifies artists whose number of songs in the dataset exceeds the average catalog size.
# HOW: Counts songs by artist, then compares each artist’s count to the average count using a subquery.
# WHY: This helps identify artists who may have more influence on aggregate results because they are represented more heavily in the data.

pd.read_sql("""
SELECT
artists,
COUNT(*) AS song_count
FROM spotify
GROUP BY artists
HAVING COUNT(*) >
(
SELECT AVG(song_total)
FROM
(
SELECT COUNT(*) AS song_total
FROM spotify
GROUP BY artists
)
)
""", conn)

,artists,song_count
0,"""Weird Al"" Yankovic",15
1,(Hed) P.E.,17
2,-M-,7
3,10 Years,4
4,10cc,10
...,...,...
6433,阿吉仔,4
6434,陳秀雯,4
6435,陳美玲,4
6436,音樂磁場,6


This query matters because artists with larger song representation may shape averages and rankings more heavily than artists with only one or two tracks. That is important context when interpreting other grouped results in the notebook.

In [15]:
# Query 9: Rank songs by Spotify popularity
# WHAT: Assigns a popularity rank across all songs in the Spotify dataset.
# HOW: Uses the RANK() window function ordered by descending popularity.
# WHY: This makes it possible to identify the top-performing songs without collapsing the full dataset through aggregation.

pd.read_sql("""
SELECT
artists,
track_name,
popularity,
RANK() OVER (ORDER BY popularity DESC) AS popularity_rank
FROM spotify
LIMIT 20
""", conn)

,artists,track_name,popularity,popularity_rank
0,Sam Smith;Kim Petras,Unholy (feat. Kim Petras),100,1
1,Sam Smith;Kim Petras,Unholy (feat. Kim Petras),100,1
2,Bizarrap;Quevedo,"Quevedo: Bzrp Music Sessions, Vol. 52",99,3
3,David Guetta;Bebe Rexha,I'm Good (Blue),98,4
4,David Guetta;Bebe Rexha,I'm Good (Blue),98,4
5,Manuel Turizo,La Bachata,98,4
6,Manuel Turizo,La Bachata,98,4
7,David Guetta;Bebe Rexha,I'm Good (Blue),98,4
8,Manuel Turizo,La Bachata,98,4
9,Manuel Turizo,La Bachata,98,4


In [16]:
# Query 10: Rank songs within each artist
# WHAT: Ranks songs by popularity within each artist’s catalog.
# HOW: Uses a partitioned window function with PARTITION BY artist and ORDER BY popularity DESC.
# WHY: This shows which songs are strongest within each artist rather than only across the full dataset.
pd.read_sql("""
SELECT
artists,
track_name,
popularity,
RANK() OVER
(
PARTITION BY artists
ORDER BY popularity DESC
) AS artist_song_rank
FROM spotify
WHERE artists IS NOT NULL
LIMIT 20
""", conn)

,artists,track_name,popularity,artist_song_rank
0,!nvite,strolling,41,1
1,!nvite,pagadoff,5,2
2,"""Puppy Dog Pals"" Cast",Puppy Dog Pals Main Title Theme,60,1
3,"""Puppy Dog Pals"" Cast",Going on a Mission,55,2
4,"""Weird Al"" Yankovic","Amish Paradise (Parody of ""Gangsta's Paradise""...",58,1
5,"""Weird Al"" Yankovic","Grapefruit Diet (Parody of ""Zoot Suit Riot"" by...",29,2
6,"""Weird Al"" Yankovic",Genius In France,26,3
7,"""Weird Al"" Yankovic",I'll Be Mellow When I'm Dead,26,3
8,"""Weird Al"" Yankovic",The Night Santa Went Crazy,26,3
9,"""Weird Al"" Yankovic",Gotta Boogie,25,6


Artist-level ranking is particularly relevant for music strategy because labels often compare songs within the same artist’s catalog when deciding which tracks deserve additional promotion or investment.

In [17]:
# Query 11: Match coverage by artist
# WHAT: Measures how many songs for each artist successfully match across Spotify and Last.fm.
# HOW: Compares artist-level counts before and after joining the two datasets.
# WHY: This helps show whether data overlap is evenly distributed across artists or whether some artists are much better represented in the merged sample.
pd.read_sql("""
SELECT
  l.artist,
  COUNT(*) AS lastfm_tracks,
  SUM(CASE WHEN s.track_name IS NOT NULL THEN 1 ELSE 0 END) AS matched_tracks,
  ROUND(
    1.0 * SUM(CASE WHEN s.track_name IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*),
    3
  ) AS match_rate
FROM lastfm l
LEFT JOIN spotify s
  ON s.track_name = l.track_name
 AND s.artists = l.artist
GROUP BY l.artist
ORDER BY match_rate ASC, lastfm_tracks DESC
LIMIT 20;
""", conn)

,artist,lastfm_tracks,matched_tracks,match_rate
0,A$AP Rocky,10,0,0.0
1,Addison Rae,10,0,0.0
2,Alex G,10,0,0.0
3,Baby Keem,10,0,0.0
4,Brent Faiyaz,10,0,0.0
5,Chappell Roan,10,0,0.0
6,Charli xcx,10,0,0.0
7,Childish Gambino,10,0,0.0
8,Daft Punk,10,0,0.0
9,Daniel Caesar,10,0,0.0


This is one of the most important SQL results in the notebook because it shows that match quality is not uniform across artists. Some artists have much stronger overlap between datasets than others, which means the merged sample may be more representative for certain artists than for others.

In [18]:
# Query 12: Artist-level performance among matched songs
# WHAT: Calculates average Spotify popularity and average Last.fm playcount for matched songs by artist.
# HOW: Joins Spotify and Last.fm, groups by artist, and summarizes both platform metrics.
# WHY: This highlights which artists appear strongest across both datasets within the matched sample.

pd.read_sql(""" SELECT
  s.artists,
  COUNT(*) AS matched_songs,
  AVG(s.popularity) AS avg_spotify_popularity,
  AVG(l.playcount) AS avg_lastfm_playcount
FROM spotify s
JOIN lastfm l
  ON s.track_name = l.track_name
 AND s.artists = l.artist
GROUP BY s.artists
HAVING COUNT(*) >= 2
ORDER BY avg_spotify_popularity DESC, avg_lastfm_playcount DESC
LIMIT 15;""", conn)

,artists,matched_songs,avg_spotify_popularity,avg_lastfm_playcount
0,Bad Bunny,8,95.500000,1.024047e+07
1,Harry Styles,3,92.000000,3.388978e+07
2,Olivia Rodrigo,5,87.400000,2.777409e+07
3,Doja Cat,6,84.333333,2.213538e+07
4,Arctic Monkeys,21,82.380952,4.022921e+07
5,Bruno Mars,4,82.250000,1.596081e+07
6,TV Girl,5,82.000000,4.034995e+07
7,Ariana Grande,5,81.800000,2.426928e+07
8,The Neighbourhood,29,81.758621,2.171531e+07
9,Mac DeMarco,2,81.000000,2.760430e+07


This query strengthens the notebook because it links artist-level popularity and engagement across both platforms. It also connects well to the broader project question of whether stronger-performing songs and artists show measurable patterns across multiple data sources.

In [19]:
# Query 13: Popularity tier summary for matched songs
# WHAT: Groups matched songs into low, medium, and high Spotify popularity tiers.
# HOW: Uses CASE WHEN to create tiers, then summarizes Last.fm listener and playcount averages for each tier.
# WHY: This helps show whether higher Spotify popularity is associated with stronger listener engagement on Last.fm.

pd.read_sql("""
SELECT
    CASE
        WHEN s.popularity < 40 THEN 'Low'
        WHEN s.popularity < 70 THEN 'Medium'
        ELSE 'High'
    END AS popularity_tier,
    COUNT(*) AS track_count,
    ROUND(AVG(l.listeners), 2) AS avg_listeners,
    ROUND(AVG(l.playcount), 2) AS avg_playcount
FROM spotify s
JOIN lastfm l
    ON LOWER(TRIM(s.artists)) = LOWER(TRIM(l.artist))
   AND LOWER(TRIM(s.track_name)) = LOWER(TRIM(l.track_name))
GROUP BY popularity_tier
ORDER BY track_count DESC
""", conn)

,popularity_tier,track_count,avg_listeners,avg_playcount
0,High,400,1892724.07,23766567.79
1,Low,205,1979288.13,21986727.86
2,Medium,122,1448509.61,15255989.25


This helps show whether songs with higher Spotify popularity also tend to have stronger Last.fm engagement.


In [20]:
# Query 14: Top matched songs by Spotify popularity and Last.fm engagement
# WHAT: Returns matched songs with strong Spotify popularity and high Last.fm listener engagement.
# HOW: Joins the datasets and orders results by popularity, listeners, and playcount.
# WHY: This highlights songs performing well across both platforms, which is more useful than looking at one platform alone.

pd.read_sql("""
SELECT
    s.artists,
    s.track_name,
    s.popularity AS spotify_popularity,
    l.listeners,
    l.playcount
FROM spotify s
JOIN lastfm l
    ON LOWER(TRIM(s.artists)) = LOWER(TRIM(l.artist))
   AND LOWER(TRIM(s.track_name)) = LOWER(TRIM(l.track_name))
WHERE s.popularity IS NOT NULL
ORDER BY s.popularity DESC, l.listeners DESC, l.playcount DESC
LIMIT 15
""", conn)

,artists,track_name,spotify_popularity,listeners,playcount
0,Bad Bunny,Tití Me Preguntó,97,937685,11617049
1,Bad Bunny,Tití Me Preguntó,97,937685,11617049
2,Bad Bunny,Tití Me Preguntó,97,937685,11617049
3,Bad Bunny,Tití Me Preguntó,97,937685,11617049
4,Harry Styles,As It Was,95,1967684,41100403
5,Joji,Glimpse of Us,94,1473199,30974304
6,Bad Bunny,Moscow Mule,94,599482,8863892
7,Bad Bunny,Moscow Mule,94,599482,8863892
8,Bad Bunny,Moscow Mule,94,599482,8863892
9,Bad Bunny,Moscow Mule,94,599482,8863892


This highlights songs doing well across both Spotify and Last.fm, which is more useful than looking at only one platform.


In [21]:
# Query 15: Artists with above-average matched-song popularity
# WHAT: Finds artists whose average Spotify popularity among matched songs is above the matched-sample average.
# HOW: First calculates artist-level averages, then compares them to the overall matched average using a subquery.
# WHY: This helps identify artists whose matched catalogs perform better than the typical matched artist.

pd.read_sql("""
SELECT
    s.artists,
    COUNT(*) AS matched_tracks,
    ROUND(AVG(s.popularity), 2) AS avg_spotify_popularity,
    ROUND(AVG(l.listeners), 2) AS avg_lastfm_listeners
FROM spotify s
JOIN lastfm l
    ON LOWER(TRIM(s.artists)) = LOWER(TRIM(l.artist))
   AND LOWER(TRIM(s.track_name)) = LOWER(TRIM(l.track_name))
GROUP BY s.artists
HAVING AVG(s.popularity) > (
    SELECT AVG(s2.popularity)
    FROM spotify s2
    JOIN lastfm l2
        ON LOWER(TRIM(s2.artists)) = LOWER(TRIM(l2.artist))
       AND LOWER(TRIM(s2.track_name)) = LOWER(TRIM(l2.track_name))
    WHERE s2.popularity IS NOT NULL
)
ORDER BY avg_spotify_popularity DESC
""", conn)

,artists,matched_tracks,avg_spotify_popularity,avg_lastfm_listeners
0,Bad Bunny,8,95.50,768583.50
1,Joji,1,94.00,1473199.00
2,Beyoncé,1,93.00,934613.00
3,Harry Styles,3,92.00,1805882.33
4,J. Cole,1,88.00,1619490.00
5,Olivia Rodrigo,5,87.40,1501869.80
6,Doja Cat,6,84.33,1502413.83
7,Tate McRae,1,84.00,864559.00
8,Kendrick Lamar,1,84.00,2236543.00
9,Drake,1,84.00,1433037.00


This helps identify artists whose matched songs are performing better than average in the dataset.

## SQL Summary

The SQL queries support the project by checking data structure, measuring overlap between Spotify and Last.fm, and exploring patterns in artist performance, song popularity, and listener engagement. The strongest results show that matched songs can be analyzed meaningfully across both platforms, especially when looking at popularity tiers, cross-platform top performers, and artists whose songs perform above average. Overall, the SQL analysis complements the Python notebook by adding another layer of evidence while also reinforcing that the merged dataset is useful but incomplete.